# PToTR Paper Case Study 02: Positron Emission Tomography (PET) Image Reconstruction

## Poisson-response Tensor-on-Tensor Regression (PToTR)

This example illustrates the use of Poisson-response Tensor-on-Tensor Regression (PToTR) for image reconstruction and appears as a case study in the following paper:

Llosa-Vite, C., & Dunlavy, D. M. (2026). 
*Poisson-response Tensor-on-Tensor Regression and Applications.* 
[arXiv:2604.07377](https://arxiv.org/abs/2604.07377).

Data used in this case study is taken from https://openneuro.org/datasets/ds003011/versions/1.2.3, under sub-03/ses-Y1MRC/anat. See the paper above for details.

**NOTE:** `ptotr-paper-02-pet-01-data.ipynb` must be run prior to running this notebook, as all data is processed in that notebook and then saved to files that are used in this notebook.

### Import dependencies

In [1]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pyttb as ttb
import numpy as np

# helper functions for this notebook
from ptotr_paper_functions import (
    fit_ptotr_sparse,
    ptotr_pet_compute_B_true,
    ptotr_pet_compute_Xs,
    ptotr_pet_compute_Ys,
)

### Load PToTR data

In [2]:
# true parameter tensor (numpy ndarray)
B_true = ptotr_pet_compute_B_true(savedata="Btrue")
B_true_tensor = ttb.tensor(B_true).permute(np.array((2, 3, 1, 0))) # B_true_tensor.shape -> (256, 256, 240, 4)
B_true = B_true.reshape(4*240, 256*256)
print(f"type(B_true_tensor): {type(B_true_tensor)}")
print(f"B_true_tensor shape: {B_true_tensor.shape}")
print(f"type(B_true): {type(B_true)}")
print(f"B_true shape: {B_true.shape}")

Loading B_true: ../data/ptotr_pet_Btrue.npy
type(B_true_tensor): <class 'pyttb.tensor.tensor'>
B_true_tensor shape: (256, 256, 240, 4)
type(B_true): <class 'numpy.ndarray'>
B_true shape: (960, 65536)


In [3]:
# responses (numpy ndarray)
Ys, Ys_ind = ptotr_pet_compute_Ys(B_true_tensor, savedata="Ys")
Ys = ttb.tensor(Ys).permute(np.array([1, 0, 2]))
print(f"type(Ys): {type(Ys)}")
print(f"Ys shape: {Ys.shape}")

Loading Ys: ../data/ptotr_pet_Ys.npy
Loading Ys_ind: ../data/ptotr_pet_Ys_ind.npy
type(Ys): <class 'pyttb.tensor.tensor'>
Ys shape: (240, 4, 41952)


In [4]:
# covariates (numpy ndarray)
Xs = ptotr_pet_compute_Xs(Ys_ind, savedata="Xs")
print(f"type(Xs): {type(Xs)}")
print(f"Xs shape: {Xs.shape}")
print(f"nnz(Xs): {Xs.nnz}")

Loading Xs: ../data/ptotr_pet_Xs.tns
type(Xs): <class 'pyttb.sptensor.sptensor'>
Xs shape: (256, 256, 41952)
nnz(Xs): 6818154


In [ ]:
%%time
rank = 2 # one of the following: 2, 5, 21, 84, 336
percents = [1,2,4,8,16]
iters = [10,120]
rmse_df, saved_imgs, final_B_by_pp = fit_ptotr_sparse(
    Y=Ys,
    X_tensor=Xs,
    B0=B_true_tensor,
    rank=rank,
    pp_values=percents,
    p1=2622,
    itint=1,
    itmax=120,
    save_iterations=iters,
    epsDivZero=1e-10,
    stoptol=-1,
    out_rmse_csv=f"../results/ptotr_rank_{rank}_rmses.csv",
    out_slices_pkl=f"../results/ptotr_rank_{rank}_slices.pkl",
)

### Plot RMSE values

In [ ]:
# Columns are iteration counts
x = [int(c) for c in rmse_df.columns]

# Plot one line per percentage of data
plt.figure(figsize=(8, 5))
for row_label in rmse_df.index:
    plt.plot(x, rmse_df.loc[row_label].values, marker='o', label=row_label)

plt.xlabel("# iterations")
plt.ylabel("RMSE")
plt.title(f"PToTR (rank={rank}) reconstruction error vs. iterations")
plt.legend(title="% of data")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Plot reconstructed images

In [ ]:
view = "coronal"   # "axial", "coronal", "sagittal"

fig, axs = plt.subplots(2, 5, figsize=(16, 6))

for i, it in enumerate(iters):
    for j, pp in enumerate(percents):
        axs[i, j].imshow(saved_imgs[(pp, it, view)][:,::-1].T, cmap="gray_r")
        axs[i, j].set_title(f"{pp}% data, {it} iters")
        axs[i, j].axis("off")

plt.suptitle(f"ML-EM reconstructions: {view} view", fontsize=14)
plt.tight_layout()
plt.show()